In [1]:
import os 
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = "/scratch/adelmou/hf_home/"
os.environ["HF_HUB_CACHE"] = "/scratch/adelmou/hf_home/hub/"

import torch 

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True


In [2]:
import speechbrain as sb
from speechbrain.utils.logger import get_logger
from speechbrain.dataio.dataloader import LoopedLoader
from speechbrain.dataio.dataloader import DataLoader
import torch
import numpy as np
from tqdm import tqdm
from abc import ABCMeta, abstractmethod
from pathlib import Path
from contextlib import ExitStack
from typing import Dict

logger = get_logger(__name__)


In [42]:

class FeaturesReader(metaclass=ABCMeta):
    @property
    @abstractmethod
    def name(self) -> str:
        ...

    @abstractmethod
    def read(
        self,
        key: str,
        left_offset_frames: int = 0,
        right_offset_frames: int = None,
    ) -> np.ndarray:
        ...

class FeaturesWriter(metaclass=ABCMeta):
    @property
    @abstractmethod
    def name(self) -> str:
        ...

    @property
    @abstractmethod
    def storage_path(self) -> str:
        ...

    @abstractmethod
    def write(self, key: str, value: np.ndarray) -> str:
        ...
    def __enter__(self):
        return self

    def __exit__(self, *args, **kwargs):
        ...


class NumpyHdf5Writer(FeaturesWriter):
    name = "numpy_hdf5"

    def __init__(self, storage_path: Path, mode: str = "w", dtype=np.float32, *args, **kwargs):
        """
        :param storage_path: Path under which we'll create the HDF5 file.
            We will add a ``.h5`` suffix if it is not already in ``storage_path``.
        :param mode: Modes supported by h5py:
            w        Create file, truncate if exists (default)
            w- or x  Create file, fail if exists
            a        Read/write if exists, create otherwise
        """
        super().__init__()
        import h5py

        p = Path(storage_path)
        self.storage_path_ = p.with_suffix(
            p.suffix + ".h5" if p.suffix != ".h5" else ".h5"
        )
        print(f"saving to {self.storage_path_}")
        self.hdf = h5py.File(self.storage_path, mode=mode)
        self.dtype = dtype

    @property
    def storage_path(self) -> str:
        return str(self.storage_path_)

    def write(self, key: str, value: np.ndarray) -> str:
        value = value.cpu().numpy().astype(self.dtype)
        self.hdf.create_dataset(key, data=value)
        return key

    def close(self) -> None:
        return self.hdf.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

class NumpyHdf5Reader(FeaturesReader):

    name = "numpy_hdf5"

    def __init__(self, storage_path: Path, *args, **kwargs):
        super().__init__()
        import h5py
        self.hdf = h5py.File(storage_path, mode="r")

    def read(
        self,
        key: str,
        left_offset_frames: int = 0,
        right_offset_frames: int = None,
    ) -> np.ndarray:
        return self.hdf[key][left_offset_frames:right_offset_frames]

    def close(self) -> None:
        return self.hdf.close()

In [45]:
class ExtractFeatures(sb.core.Brain):
    def compute_features(self, batch):
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        feats = torch.randn(wavs.shape[0], 1024)
        batch_size = wavs.shape[0]
        codecs = torch.randint(0, 2048, (batch_size, 32, 512))
        
        return [{"id": batch.id[i], "feats": feats[i], "codecs": codecs[i]} for i in range(batch_size)]
    
    def cache_features(
        self,
        writers: Dict[str, FeaturesWriter],
        dataset,
        loader_kwargs={},
        progressbar=None,
    ):
        assert writers is not None, "Feature cache manager must be provided"
        
        if not (
            isinstance(dataset, DataLoader)
            or isinstance(dataset, LoopedLoader)
        ):
            dataset = self.make_dataloader(
                dataset, stage=sb.Stage.TEST, **loader_kwargs
            )
        
        if progressbar is None:
            progressbar = not self.noprogressbar

        # Only show progressbar if requested and main_process
        enable = progressbar and sb.utils.distributed.if_main_process()
        feature_types = list(writers.keys())
        with ExitStack() as stack:
            open_writers = {ft: stack.enter_context(writers[ft]) for ft in writers.keys()}
            for data in self._cache_features(
                dataset=dataset,
                enable=enable,
            ):
                for item in data:
                    for ft in feature_types:
                        open_writers[ft].write(item['id'], item[ft])


    def _cache_features(self, dataset, enable):
        for batch in tqdm(dataset, dynamic_ncols=True, disable=not enable, colour=self.tqdm_barcolor["valid"]):
            extracted_features = self.compute_features(batch)
            yield extracted_features
    

In [5]:
import os 
from librispeech_prepare import prepare_librispeech  # noqa
prepare_librispeech(
    data_folder= os.environ["SLURM_TMPDIR"] + "/LibriSpeech",
    tr_splits= [],
    dev_splits= ['dev-clean'],
    te_splits= [],
    save_folder= os.environ["SLURM_TMPDIR"] + "/save",
    merge_lst= [],
    merge_name= "train.csv",
    skip_prep= False,
)

In [6]:
def dataio_prepare():
    """This function prepares the datasets to be used in the brain class.
    It also defines the data processing pipeline through user-defined functions.
    """
    data_folder = os.environ["SLURM_TMPDIR"] + "/save"

    train_data = sb.dataio.dataset.DynamicItemDataset.from_csv(
        csv_path=os.environ["SLURM_TMPDIR"] + "/save/dev-clean.csv",
        replacements={"data_root": data_folder},
    )

    datasets = [train_data]

    # 2. Define audio pipeline:
    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav):
        sig = sb.dataio.dataio.read_audio(wav)
        return sig

    sb.dataio.dataset.add_dynamic_item(datasets, audio_pipeline)

    # 4. Set output:
    sb.dataio.dataset.set_output_keys(
        datasets,
        ["id", "sig"],
    )

    return (
        train_data,
    )

train_data, = dataio_prepare()

In [62]:
feature_types = ['feats', 'codecs']
feature_dtypes = [np.float32, np.int16]
writers = {
    ft: NumpyHdf5Writer(
        storage_path=os.path.join(os.environ['SLURM_TMPDIR'], f"save/dev-clean_{ft}.h5"),
        dtype=feature_dtypes[idx]
    )
    for idx, ft in enumerate(feature_types)
}

extractor = ExtractFeatures()
dataloader = sb.dataio.dataloader.make_dataloader(
    train_data, **{"batch_size": 12}
)

The model has no parameters!


saving to /localscratch/adelmou.45628354.0/save/dev-clean_feats.h5
saving to /localscratch/adelmou.45628354.0/save/dev-clean_codecs.h5


In [63]:
extractor.cache_features(writers, dataloader)

['feats', 'codecs']


100%|█████████████████████████████████████████████████████████████████████████| 226/226 [00:06<00:00, 32.77it/s]


In [65]:
reader_manager = NumpyHdf5Reader(storage_path=os.environ["SLURM_TMPDIR"] + "/save/dev-clean_codecs.h5")
for idx, i in enumerate(train_data):
    print(i['id'])
    print(reader_manager.read(i['id']))
    if idx > 10:
        break
reader_manager.close()

1919-142785-0031
[[1806 1032  839 ...  868 1507 1727]
 [1832  354  538 ... 1514  977 1169]
 [1318  849  498 ...  999  229  730]
 ...
 [ 485 1554  851 ... 1849 1963  848]
 [2040 1603 1902 ... 1412 2020 1082]
 [ 106  781  789 ...  759    1 1455]]
1919-142785-0000
[[ 418 1322 1009 ... 1811  686 1167]
 [1712  349 1521 ...  786 1513 1313]
 [ 101 1994  377 ... 1499  806 1189]
 ...
 [  89 1643 1133 ... 1675  336 1183]
 [1963 1293 1400 ... 1242 1735 1390]
 [1894  192 1896 ...  186   49 1170]]
1919-142785-0029
[[1973  392 1650 ...  865    4 1932]
 [ 769 1413 1645 ... 1268 1405  632]
 [  63  165  954 ... 1603  865 1624]
 ...
 [1673 1961  382 ...  245  945  804]
 [ 387  815  915 ...  708 1438 1332]
 [ 650 1254  150 ... 1308 1199 1470]]
1919-142785-0062
[[ 502 1086 1552 ... 1593 1475  564]
 [1617  264  183 ...  633  526 1142]
 [ 248  151 1228 ... 1836  479  740]
 ...
 [ 268  375 1235 ...  292   88  546]
 [  90  557 1460 ...  895 1223 2015]
 [1847 1049 1794 ...  116  601 1073]]
1919-142785-0053
[[ 